# Timetables

The timetable generator (`timetable_generator` in `flatland.envs.timetable_generators`, see [Scenario Generation](../scenario_generation)) gives each
agent an earliest departure and a latest arrival at each waypoint of its line: the start, the intermediate stops and the target. The start has no latest
arrival, the target no earliest departure.

```{important}
A single agent without malfunctions, with the default `acceleration_delta=1` and `braking_delta=-1`, can keep its timetable at any max speed $v$: driving
the shortest route through its stops at max speed, it enters each stop at least $\lceil L/v \rceil - \lceil (L-1)/v \rceil \ge 1$ steps before the
stop's earliest departure ($L$: the segment's length, see below), halts there, starts moving again at the earliest departure and reaches its target by its
latest arrival, without penalty. The latest arrivals are the same times, shifted later by a buffer.

Without rounding, for max speed $v = 1/n$ ($n$ = 1, 2, 3, ...; all speeds of `env_generator()`'s default `speed_ratios`) and whole edge lengths: the
agent enters each stop exactly $n$ steps before its earliest departure (1 step at max speed 1), and
$\delta_i = \max(\delta_0, 1) + n \sum_{j \le i} L_j$, $\alpha_i = \alpha_J - n \sum_{j > i} L_j$.
```

The times are what the agent can do on its own. With other agents on the network, keeping them is the agent's problem: conflicts at junctions and on
single tracks are not planned for.

## Inputs and constants

`timetable_generator(agents, distance_map, agents_hints, np_random)` has no parameters of its own. It takes what `RailEnv.reset()` passes:

| input          | used for                                                                                                                    |
|:---------------|:----------------------------------------------------------------------------------------------------------------------------|
| `agents`       | each agent's waypoints (start, stops with their entry points, target) and max speed $v$                                     |
| `distance_map` | the rail, with its edge lengths, for the segment lengths; the grid's width and height for the episode length's upper bound |
| `agents_hints` | the number of cities (`city_positions`), for the episode length's upper bound; 2 without hints                              |
| `np_random`    | draws each agent's earliest departure at the start                                                                          |

Its constants are hard-coded:

| constant                           | value | meaning                                                                                                   |
|:-----------------------------------|:------|:----------------------------------------------------------------------------------------------------------|
| `travel_buffer_multiplier`         | 1.3   | the latest arrival at the target allows 1.3 times the agent's travel time $T$                             |
| `mean_shortest_path_multiplier`    | 0.2   | plus 0.2 times the mean travel time $\bar{T}$ of all agents, for the latest arrival and the episode length |
| `new_max_episode_steps_multiplier` | 1.5   | the episode length allows 1.5 times the longest travel time (must be more than 1.3)                       |
| `old_max_episode_steps_multiplier` | 3.0   | upper bound of the episode length: 3 times $4 \cdot 2 \cdot (\text{width} + \text{height} + \text{agents} / \text{cities})$ |
| `timedelay_factor`, `alpha`        | 4, 2  | the factors $4 \cdot 2$ of that upper bound                                                               |
| `end_buffer_multiplier`            | 0.05  | no latest arrival in the last 5% of the episode                                                           |

The acceleration and braking deltas of the env are not among the inputs: the timetable assumes the defaults (see below).

## How the times are computed

The generator plans each agent's line along one route: the shortest route from the start through all intermediate stops to the target. A stop is served
at one of its entry points (position and direction), so the route chooses each stop's entry point over the whole line: the agent leaves a stop in the
direction it arrived in.

- **Segment length** $L_i$: the part of the route from waypoint $i-1$ to waypoint $i$, as the number of cells it crosses plus 1 (the sum of the edge
  lengths plus 1, rounded up to whole steps). The +1 stands for the step onto the map, or for the halt at a stop.
- **Travel time per segment**: $\lceil L_i / v \rceil$ steps, for max speed $v$, rounded up per segment: the agent starts each segment from a halt, and
  each segment gets the whole steps the agent needs from there. Rounding the sum of the segments instead can leave a segment one step short.
- **Earliest departures**: the start's earliest departure $\delta_0$ is drawn at random. The earliest departure at stop $i$ adds the travel times of the
  segments up to it, counted from $\max(\delta_0, 1)$, the first step in which the agent can enter the map:
  $\delta_i = \max(\delta_0, 1) + \sum_{j \le i} \lceil L_j / v \rceil$.
- **Latest arrival at the target**: $\alpha_J = \delta_0 + \lceil 1.3\, T + 0.2\, \bar{T} \rceil$, with the travel time $T = \sum_j L_j / v$ (not rounded)
  and $\bar{T}$ the mean of $T$ over all agents. The 30% of $T$ and 20% of $\bar{T}$ are the buffer.
- **Latest arrivals at the stops**: the latest arrival at the target minus the travel times of the segments after the stop:
  $\alpha_i = \alpha_J - \sum_{j > i} \lceil L_j / v \rceil$. So the latest arrivals are the earliest times shifted later, by the same buffer for all
  waypoints (one step less where $\delta_0 = 0$).
- **Earliest departure at the start**: $\delta_0$ is drawn uniformly from $0$ to $\max(\alpha_{\max} - \lceil 1.3\, T + 0.2\, \bar{T} \rceil, 1) - 1$, where
  $\alpha_{\max}$ is the episode length minus its last 5%, so the latest arrival at the target falls before $\alpha_{\max}$ where the episode is long
  enough.
- **Episode length**: `max_episode_steps` $= \min(\lfloor \lceil 1.5 \max T \rceil + 0.2\, \bar{T} \rfloor,\ 3 \cdot \lfloor 4 \cdot 2 \cdot (\text{width} +
  \text{height} + \text{agents} / \text{cities}) \rfloor)$, with the maximum over the agents.

What the times mean for the agent:

- The agent **arrives** at a stop in the step it enters the stop's cell. Driving at max speed $v$ from the start of the segment, it needs
  $\lceil (L_i - 1)/v \rceil$ steps for the $L_i - 1$ cells, so it arrives at least $\lceil L_i/v \rceil - \lceil (L_i - 1)/v \rceil$ steps before the
  earliest departure: exactly 1 at max speed 1, $n$ at $1/n$ (the +1 per segment, divided by the speed). At other speeds, such as 3/4, the agent may halt
  further into a stop's cell and arrive at the next stop one step earlier.
- The agent can **halt** at the stop in the step it enters it (`STOP_MOVING` in that step), and **start moving** again in the step of the earliest
  departure. It then enters the next cell $\lceil 1/v \rceil$ steps later.
- The agent enters the map in the step of its earliest departure, at the earliest in step 1: an earliest departure of 0 behaves as 1.

The example below is a straight line of 12 cells: the agent starts in column 1 heading east, stops in columns 4 and 7 and ends in column 10. Each segment
crosses 3 cells, so $L_i = 4$. The table shows the times the generator plans for max speeds 1, 3/4 and 1/2, and the steps the agent needs.

In [ ]:
from fractions import Fraction
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from flatland.core.grid.grid4 import Grid4TransitionsEnum
from flatland.envs.grid.rail_env_grid import RailEnvTransitions
from flatland.envs.rail_env import RailEnv
from flatland.envs.rail_env_action import RailEnvActions
from flatland.envs.rail_generators import rail_from_grid_transition_map
from flatland.envs.rail_grid_transition_map import RailGridTransitionMap
from flatland.envs.rail_trainrun_data_structures import Waypoint
from flatland.envs.rewards import DefaultRewards
from flatland.envs.timetable_utils import Line

N_CELLS = 12
START, STOPS, TARGET = 1, [4, 7], 10  # columns of the single row; the agent heads east
SPEEDS = [Fraction(1), Fraction(3, 4), Fraction(1, 2)]
INK, MUTED, GRID = "#1f2937", "#6b7280", "#e5e7eb"
DISTANCE_WIDTH, SPEED_WIDTH = 2, 1.5  # the lines and their y axes
Y_TOP, ROW_HEIGHT = TARGET - START + 4.4, 1.2  # top of the distance axis, height of a row of the windows' labels
AGENT_COLOR, SPEED_COLOR, WINDOW_COLOR, DRIVE_COLOR, STOP_COLOR = "#1e3a8a", "#9ca3af", "#38bdf8", "#16a34a", "#dc2626"


def make_env(max_speed):
    """A straight line of N_CELLS cells with dead ends, one agent with two intermediate stops and the given max speed."""
    transitions = RailEnvTransitions()
    straight = transitions.rotate_transition(transitions.transition_list[1], 90)
    dead_end = transitions.transition_list[7]
    rail = RailGridTransitionMap(width=N_CELLS, height=1, transitions=transitions)
    rail.grid = np.array([[transitions.rotate_transition(dead_end, 270)] + [straight] * (N_CELLS - 2)
                          + [transitions.rotate_transition(dead_end, 90)]], dtype=np.uint16)
    hints = {"agents_hints": {"city_positions": [(0, 0), (0, N_CELLS - 1)],
                              "train_stations": [[((0, 0), 0)], [((0, N_CELLS - 1), 0)]], "city_orientations": [1, 3]}}

    def line_generator(rail, number_of_agents, hints, num_resets, np_random):
        east = Grid4TransitionsEnum.EAST
        return Line(agent_waypoints={0: [[Waypoint((0, START), east)]] + [[Waypoint((0, c), east)] for c in STOPS]
                                        + [[Waypoint((0, TARGET), None)]]},
                    agent_speeds=[max_speed])

    rewards = DefaultRewards()
    env = RailEnv(width=N_CELLS, height=1, number_of_agents=1, rail_generator=rail_from_grid_transition_map(rail, hints),
                  line_generator=line_generator, rewards=rewards)
    env.reset(random_seed=3)
    return env, rewards


def drive(env, rewards):
    """Drives the agent east: it brakes in the step it enters a stop, halts until the step of the stop's earliest
    departure, and starts moving again in that step. Returns per elapsed_steps the speed and the distance travelled
    from the start cell (None off the map), the action given at each elapsed_steps (into the step to the next one),
    the steps it enters each waypoint's cell and starts moving at each stop, and the penalties of DefaultRewards at
    the end."""
    agent = env.agents[0]
    stops = list(zip(STOPS, agent.waypoints_earliest_departure[1:-1]))
    speed, travelled, actions = [None], [None], []
    entered, started = {}, {}
    while not env.dones["__all__"] and env._elapsed_steps < env._max_episode_steps:
        action = RailEnvActions.MOVE_FORWARD
        if agent.current_entry_point is not None and stops:
            column, ed = stops[0]
            speed_counter = agent.speed_counter
            here = agent.current_entry_point[0][1]
            if here == column:
                if env._elapsed_steps + 1 < ed:
                    action = RailEnvActions.STOP_MOVING
                else:
                    started[column] = env._elapsed_steps + 1
                    stops.pop(0)
            elif here + 1 == column and speed_counter.distance + speed_counter.speed >= speed_counter.edge_length:
                # crossing into the stop in this step: brake, so the agent halts there
                action = RailEnvActions.STOP_MOVING
        before = agent.current_entry_point
        actions.append(action)
        env.step({0: action})
        entry_point = agent.current_entry_point
        if entry_point is not None and entry_point != before:
            entered.setdefault(entry_point[0][1], env._elapsed_steps)
        if agent.arrival_time is not None:
            entered.setdefault(TARGET, agent.arrival_time)
            speed.append(0)
            travelled.append(TARGET - START)
        else:
            speed.append(None if entry_point is None else agent.speed_counter.speed)
            travelled.append(None if entry_point is None else entry_point[0][1] - START + agent.speed_counter.distance)
    penalties = rewards._proxy._agent_done_or_max_episode_steps_reward(agent, env.distance_map, env._elapsed_steps,
                                                                       rewards._proxy.empty())
    return speed, travelled, actions, entered, started, {k: v for k, v in penalties.items() if v}


runs = {}
for max_speed in SPEEDS:
    env, rewards = make_env(max_speed)
    agent = env.agents[0]
    speed, travelled, actions, entered, started, penalties = drive(env, rewards)
    # the guarantee: each stop entered before its earliest departure, started at it, the target by its latest arrival
    eds, las = agent.waypoints_earliest_departure, agent.waypoints_latest_arrival
    # each segment crosses 3 cells, so L = 4: the agent enters a stop at least ceil(4 / v) - ceil(3 / v) steps early,
    # exactly at max speed 1/n
    margin = math.ceil(4 / max_speed) - math.ceil(3 / max_speed)
    exact = (1 / max_speed).denominator == 1
    assert all(ed - entered[c] == margin if exact else ed - entered[c] >= margin for c, ed in zip(STOPS, eds[1:-1])), \
        (entered, eds, margin)
    assert [started[c] for c in STOPS] == eds[1:-1], (started, eds)
    assert agent.arrival_time <= las[-1] and not penalties, (agent.arrival_time, las, penalties)
    runs[max_speed] = dict(eds=eds, las=las, speed=speed, travelled=travelled, actions=actions, entered=entered,
                           started=started,
                           max_episode_steps=env._max_episode_steps, arrival=agent.arrival_time)

In [ ]:
rows = []
for max_speed, run in runs.items():
    eds, las = run["eds"], run["las"]
    first = max(eds[0], 1)
    rows.append({
        "max speed v": str(max_speed),
        "L / v": str(Fraction(4) / max_speed),
        "⌈L / v⌉ (planned per segment)": math.ceil(Fraction(4) / max_speed),
        "steps to the next stop, then 1 halt": f"{math.ceil(Fraction(3) / max_speed)} + 1",
        "earliest departures δ": eds[:-1],
        "agent enters the map, stops 1, 2 and target": [first] + [run["entered"][c] for c in STOPS] + [run["arrival"]],
        "agent starts at stops 1, 2": [run["started"][c] for c in STOPS],
        "latest arrivals α": las[1:],
        "max_episode_steps": run["max_episode_steps"],
    })
pd.DataFrame(rows).style.hide(axis="index")

## Speed profiles

The plots show the example's agent at max speeds 1, 3/4 and 1/2 over `elapsed_steps`: the distance travelled from the start cell (in cells, as
steps: a value set by the step to `elapsed_steps = k` holds from $k$ on, dark blue, left axis) and the speed (grey, right axis). The speed changes
during a step: its line goes straight from the value before the step, at $k-1$, to the value after it, at $k$. So the agent has its max speed only at the
end of the step it accelerates in, and speed 0 only at the end of the step it brakes in. Off the map, the speed is shown as 0. The horizontal lines are the waypoints' cells.
The timetable is drawn as light blue boxes over the full height, one per waypoint and labelled with its cell: from the earliest departure $\delta$ to the
latest arrival $\alpha$ at a stop, a single line at the start (the agent enters the map at $\max(\delta_0, 1)$) and at the target. Below each plot are the
actions, at the `elapsed_steps` they are given at, into the step to the next one: green for driving (`MOVE_FORWARD`, `MOVE_LEFT`, `MOVE_RIGHT`), red for
`STOP_MOVING`.

- The agent brakes (red) in the step it enters a stop's cell, halts there (the flat part, more red) and drives on (green) in the step of the stop's
  earliest departure, the left edge of the stop's box.
- At max speed 1, the agent enters a stop exactly one step before its earliest departure: the timetable is tight. At 3/4 and 1/2, the +1 per segment is
  divided by the speed too, so the agent has 1 or 2 steps to spare.
- The right edges of the boxes, the latest arrivals, are the earliest departures shifted later by the buffer: an agent that enters a stop by its latest
  arrival, halts and starts in the next step can still reach the next stop by its latest arrival.

In [ ]:
x_max = max(run["max_episode_steps"] for run in runs.values())
names = ["start", "stop 1", "stop 2", "target"]
columns = [START] + STOPS + [TARGET]
# one group per max speed: the plot with its action strip right below it, more space between the groups
fig = plt.figure(figsize=(9, 3.9 * len(runs)))
groups = fig.add_gridspec(len(runs), 1, hspace=0.3)
axes = []
for group in groups:
    inner = group.subgridspec(2, 1, height_ratios=[5, 0.5], hspace=0.02)
    ax = fig.add_subplot(inner[0], sharex=axes[0] if axes else None)
    axes += [ax, fig.add_subplot(inner[1], sharex=ax)]
for (ax, ax_a), (max_speed, run) in zip(zip(axes[::2], axes[1::2]), runs.items()):
    # time windows: from the earliest departure to the latest arrival at each stop, over the full height; the start has
    # only an earliest departure (the agent enters the map from step 1), the target only a latest arrival
    windows = [(max(run["eds"][0], 1), max(run["eds"][0], 1))] + list(zip(run["eds"][1:-1], run["las"][1:-1])) \
              + [(run["las"][-1], run["las"][-1])]
    for ed, la in windows:
        if ed < la:
            ax.axvspan(ed, la, color=WINDOW_COLOR, alpha=0.10, zorder=0)
        for x in {ed, la}:
            ax.axvline(x, color=WINDOW_COLOR, linewidth=1, zorder=1)
    # the windows' labels and arrows at the top, each in the highest row where it overlaps no other label or arrow
    ax.set_xlim(0, x_max + 1)
    ax.set_ylim(-0.5, Y_TOP)
    renderer = fig.canvas.get_renderer()
    occupied = []
    for name, column, (ed, la) in zip(names, columns, windows):
        times = (f"δ = {run['eds'][0]}" if name == "start" else f"α = {la}" if name == "target"
                 else f"δ = {ed} to α = {la}")
        right = name == "target"
        text = ax.text(ed - 0.15 if right else ed + 0.15, 0, f"{name}, cell {(0, column)}: {times}", fontsize=7.5,
                       color="#0369a1", va="bottom", ha="right" if right else "left", zorder=5)
        box = text.get_window_extent(renderer).transformed(ax.transData.inverted())
        left, right_end = min(box.x0, ed) - 0.3, max(box.x1, la) + 0.3
        row = next(r for r in range(10) if all(r != o_row or right_end < o_left or left > o_right
                                              for o_left, o_right, o_row in occupied))
        occupied.append((left, right_end, row))
        y = Y_TOP - 1.2 - ROW_HEIGHT * row
        text.set_y(y + 0.3)
        if ed < la:
            ax.annotate("", xy=(la, y), xytext=(ed, y), zorder=5,
                        arrowprops=dict(arrowstyle="<->", color="#0369a1", linewidth=1, shrinkA=0, shrinkB=0))
    ks = range(len(run["travelled"]))
    ax_v = ax.twinx()
    # the speed changes during a step: a straight line from its value before the step (k - 1) to the one after it (k)
    ax_v.plot(ks, [float(v) if v is not None else 0 for v in run["speed"]], color=SPEED_COLOR, linewidth=SPEED_WIDTH)
    ax_v.set_ylim(0, 3)
    ax_v.set_yticks([0, float(max_speed)], ["0", str(max_speed)])
    ax_v.tick_params(colors=SPEED_COLOR, labelsize=8)
    ax_v.set_ylabel("speed", color=SPEED_COLOR, fontsize=8)
    travelled = [float(d) if d is not None else np.nan for d in run["travelled"]]
    ax.step(ks, travelled, where="post", color=AGENT_COLOR, linewidth=DISTANCE_WIDTH, zorder=3)
    for column in columns:
        ax.axhline(column - START, color=GRID, linewidth=0.8, zorder=0)
    ax.set_yticks([c - START for c in columns])
    ax.set_ylabel("distance travelled\n(cells)", color=AGENT_COLOR, fontsize=9)
    ax.tick_params(axis="y", colors=AGENT_COLOR)
    for spine in (ax.spines["left"], ax_v.spines["left"]):
        spine.set_color(AGENT_COLOR)
        spine.set_linewidth(DISTANCE_WIDTH)
    ax.spines["right"].set_visible(False)
    ax_v.spines["right"].set_color(SPEED_COLOR)
    ax_v.spines["right"].set_linewidth(SPEED_WIDTH)
    ax.tick_params(axis="y", width=DISTANCE_WIDTH)
    ax_v.tick_params(axis="y", width=SPEED_WIDTH)
    ax.grid(axis="x", color=GRID, linewidth=0.6)
    ax.set_title(f"max speed {max_speed}", loc="left", fontsize=10, color=INK, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax_v.spines["top"].set_visible(False)
    # actions, at the elapsed_steps they are given at (into the step to the next elapsed_steps)
    for k, action in enumerate(run["actions"]):
        stop = action == RailEnvActions.STOP_MOVING
        ax_a.plot(k, 0, marker="s" if stop else "^", color=STOP_COLOR if stop else DRIVE_COLOR, markersize=4.5,
                  linestyle="none")
    ax_a.set_ylim(-1, 1)
    ax_a.set_yticks([0], ["action"], fontsize=8)
    ax_a.tick_params(axis="y", length=0)
    ax_a.grid(axis="x", color=GRID, linewidth=0.6)
    for side in ("top", "left", "right"):
        ax_a.spines[side].set_visible(False)
    ax.tick_params(axis="x", labelbottom=False)
    ax_a.set_xticks(range(0, x_max + 2, 2))
    ax_a.tick_params(axis="x", labelsize=8, colors=INK)
    ax_a.set_xlabel("elapsed_steps", color=INK, fontsize=8, labelpad=1)
handles = [plt.Line2D([], [], color=AGENT_COLOR, linewidth=DISTANCE_WIDTH, label="distance travelled"),
           plt.Line2D([], [], color=SPEED_COLOR, linewidth=SPEED_WIDTH, label="speed (right axis)"),
           plt.matplotlib.patches.Patch(facecolor=WINDOW_COLOR, alpha=0.25, edgecolor=WINDOW_COLOR,
                                        label="earliest departure δ to latest arrival α"),
           plt.Line2D([], [], color=DRIVE_COLOR, marker="^", linestyle="none", label="MOVE_FORWARD / LEFT / RIGHT"),
           plt.Line2D([], [], color=STOP_COLOR, marker="s", linestyle="none", label="STOP_MOVING")]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.07), ncol=3, frameon=False, fontsize=9)
plt.show()

## What is not guaranteed

- **Other agents and malfunctions**: the times are planned for each agent alone; delays from conflicts or malfunctions are only absorbed by the buffer of
  the latest arrivals.
- **Other acceleration and braking deltas**: with `acceleration_delta` below 1, the agent needs more than one step to reach its max speed after each halt,
  and with `braking_delta` above -1 more than one step to halt. The timetable generator doesn't get the env's deltas and does not plan for them
  ([flatland-rl#587](https://github.com/flatland-association/flatland-rl/issues/587)).
- **Routes through the target**: the agent is done the first time it enters its target. If the shortest route to a stop passes the target, the agent is
  done before serving the stop ([flatland-rl#557](https://github.com/flatland-association/flatland-rl/issues/557)).
- **Unreachable stops**: a stop none of whose entry points can be reached in its listed direction is planned in any direction at its position, and a
  segment that can't be reached at all counts 0. Such a stop can't be served.

How keeping or missing the times is rewarded is described in [Rewards](rewards).

```{admonition} Code reference
The timetable generator is defined in [envs/timetable_generators.py](https://github.com/flatland-association/flatland-rl/blob/main/flatland/envs/timetable_generators.py).
```